# WASH Indicators -> Incidence Forecast — Interactive Demo

Simple notebook interface: pick a country from a dropdown, see its incidence history and the model's next-year prediction, with baselines for context.

Requires `MERGED_normalized.csv` and `tuned_results.json` in the same folder, and `catboost`, `ipywidgets`, `matplotlib` installed.

In [7]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from catboost import CatBoostRegressor

SEED = 42
MODEL_KEY = "CatBoost (Normalized)"  # best-performing model from model_comparison_final.csv

## 1. Load data and train the model (same pipeline as `Model_Training.ipynb`)

In [8]:
df = pd.read_csv("MERGED_normalized.csv")
df['Incidence_next_log'] = np.log1p(df['Incidence_next'])

model_df = pd.get_dummies(df, columns=['WHO_Region'], dummy_na=True)
exclude_cols = ['Country', 'Year', 'Autochthonous_Cases', 'Population',
                'Incidence', 'Incidence_next', 'Incidence_next_log']
feature_cols = [c for c in model_df.columns if c not in exclude_cols]

X = model_df[feature_cols]
y_log = model_df['Incidence_next_log']
y_orig = df['Incidence_next']
years = df['Year']

train_mask = years <= 2019
test_mask = years > 2019

with open("tuned_results.json") as f:
    tuned = json.load(f)
params = tuned[MODEL_KEY]['best_params']

model = CatBoostRegressor(random_state=SEED, verbose=0, bootstrap_type='Bernoulli', **params)
model.fit(X[train_mask], y_log[train_mask])

pred_log = model.predict(X[test_mask])
pred = np.clip(np.expm1(pred_log), a_min=0, a_max=None)
rmse = np.sqrt(np.mean((y_orig[test_mask].values - pred) ** 2))
mean_rmse = np.sqrt(np.mean((y_orig[test_mask].values - y_orig[train_mask].mean()) ** 2))

print(f"Temporal test RMSE:        {rmse:.1f}")
print(f"Naive mean-predict RMSE:   {mean_rmse:.1f}")
print("Model beats the naive baseline only modestly -- treat output as a trend signal.")

/var/folders/5w/2zqvdwxd2nq_f6vfhvcfy4fw0000gn/T/ipykernel_2637/2917104902.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Incidence_next_log'] = np.log1p(df['Incidence_next'])


Temporal test RMSE:        137.7
Naive mean-predict RMSE:   137.4
Model beats the naive baseline only modestly -- treat output as a trend signal.


## 2. Interactive country picker

In [9]:
country_dropdown = widgets.Dropdown(
    options=sorted(df['Country'].dropna().unique()),
    value='Afghanistan' if 'Afghanistan' in df['Country'].values else df['Country'].iloc[0],
    description='Country:'
)
output = widgets.Output()

def render(change=None):
    with output:
        clear_output(wait=True)
        country = country_dropdown.value
        country_df = df[df['Country'] == country].sort_values('Year')
        latest_idx = country_df.index[-1]
        latest_year = int(country_df.loc[latest_idx, 'Year'])
        latest_incidence = country_df.loc[latest_idx, 'Incidence']

        row = model_df.loc[[latest_idx], feature_cols]
        pred_log_single = model.predict(row)[0]
        pred_single = float(np.clip(np.expm1(pred_log_single), a_min=0, a_max=None))

        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.plot(country_df['Year'], country_df['Incidence'], marker='o', label='Historical incidence')
        ax.scatter([latest_year + 1], [pred_single], color='red', zorder=5, label='Predicted next year')
        ax.set_title(f'{country}: incidence history and forecast')
        ax.set_xlabel('Year')
        ax.set_ylabel('Incidence per 100,000')
        ax.legend()
        plt.show()

        print(f"Predicted incidence for {latest_year + 1}: {pred_single:.1f}")
        if pd.notna(latest_incidence):
            print(f"Persistence baseline (repeat {latest_year} value): {latest_incidence:.1f}")
        else:
            print("Persistence baseline: unavailable (missing current-year incidence)")
        print(f"Global mean-predict baseline: {df['Incidence_next'].mean():.1f}")

country_dropdown.observe(render, names='value')
display(country_dropdown, output)
render()

Dropdown(description='Country:', options=('Afghanistan', 'Angola', 'Australia', 'Bahamas', 'Bahrain', 'Banglad…

Output()